<a href="https://colab.research.google.com/github/BG-bibek/BG-bibek/blob/main/Copy_of_week10_1_DL_Time_series_in_Tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instructions for students:

Change the colab runtime type to 'T4 GPU' for this notebook.

Run each cell in this notebook, one by one, reading the comment above the cell first. This will aid in understanding what the code is doing.

After running all cells, you can compare the MAE of your deep learning model with that of other models or baseline methods. Additionally, you can fine-tune hyperparameters, adjust model architecture, or preprocess the data differently to try and improve the MAE.

# Deep learning for timeseries

## A temperature-forecasting example

In this example, we aim to predict the temperature 24 hours in the future.

In [ ]:
!wget https://s3.amazonaws.com/keras-datasets/jena_climate_2009_2016.csv.zip
!unzip jena_climate_2009_2016.csv.zip

**Inspecting the data of the Jena weather dataset**

Jena Climate is weather timeseries dataset recorded at the Weather Station of the Max Planck Institute for Biogeochemistry in Jena, Germany.

Jena Climate dataset is made up of 14 different quantities (such air temperature, atmospheric pressure, humidity, wind direction, and so on) were recorded every 10 minutes, over several years. This dataset covers data from January 1st 2009 to December 31st 2016.

For more information, please go to:
https://www.kaggle.com/datasets/mnassrib/jena-climate

In [ ]:
import os
fname = os.path.join("jena_climate_2009_2016.csv")

with open(fname) as f:
    data = f.read()

lines = data.split("\n")
header = lines[0].split(",")
lines = lines[1:]
print(header)
print(len(lines))

**Parsing the data**

In [ ]:
import numpy as np
temperature = np.zeros((len(lines),))
raw_data = np.zeros((len(lines), len(header) - 1))
for i, line in enumerate(lines):
    values = [float(x) for x in line.split(",")[1:]]
    temperature[i] = values[1]
    raw_data[i, :] = values[:]

**Plotting the temperature timeseries**

In [ ]:
from matplotlib import pyplot as plt
plt.plot(range(len(temperature)), temperature)

**Plotting the first 10 days of the temperature timeseries**

In [ ]:
plt.plot(range(1440), temperature[:1440])

**Computing the number of samples we'll use for each data split**

In [ ]:
num_train_samples = int(0.5 * len(raw_data))
num_val_samples = int(0.25 * len(raw_data))
num_test_samples = len(raw_data) - num_train_samples - num_val_samples
print("num_train_samples:", num_train_samples)
print("num_val_samples:", num_val_samples)
print("num_test_samples:", num_test_samples)

### Preparing the data

**Normalizing the data**

Normalizing data is crucial for deep learning for several reasons:

Improved convergence: Normalizing data helps in speeding up the convergence of gradient-based optimization algorithms, such as stochastic gradient descent (SGD). When features are on different scales, the optimization process can be slowed down, as the gradients corresponding to different features may vary significantly. Normalization scales all features to a similar range, which helps the optimization algorithm converge faster.

Better generalization: Normalizing data can prevent the model from being biased towards features with larger magnitudes. Without normalization, features with larger scales may dominate the learning process, leading to a model that performs poorly on unseen data. Normalization ensures that all features contribute equally to the learning process, resulting in a model that generalizes better to new data.

Stability: Normalizing data helps in stabilizing the training process. Deep neural networks are sensitive to the scale of input features, and normalizing data prevents extreme values from causing numerical stability issues during training. This stability ensures that the model learns meaningful patterns from the data without being affected by outliers or extreme values.

In [ ]:
mean = raw_data[:num_train_samples].mean(axis=0)
raw_data -= mean
std = raw_data[:num_train_samples].std(axis=0)
raw_data /= std

In [ ]:
mean

In [ ]:
std

Create sequences of input data, where each sequence contains a window of historical temperature data along with corresponding target temperatures.

In [ ]:
import numpy as np
from tensorflow import keras
int_sequence = np.arange(10)
dummy_dataset = keras.utils.timeseries_dataset_from_array(
    data=int_sequence[:-3],
    targets=int_sequence[3:],
    sequence_length=3,
    batch_size=2,
)

for inputs, targets in dummy_dataset:
    for i in range(inputs.shape[0]):
        print([int(x) for x in inputs[i]], int(targets[i]))

**Instantiating datasets for training, validation, and testing**

In [ ]:
sampling_rate = 6
sequence_length = 120
delay = sampling_rate * (sequence_length + 24 - 1)
batch_size = 256

train_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    shuffle=True,
    batch_size=batch_size,
    start_index=0,
    end_index=num_train_samples)

val_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    shuffle=True,
    batch_size=batch_size,
    start_index=num_train_samples,
    end_index=num_train_samples + num_val_samples)

test_dataset = keras.utils.timeseries_dataset_from_array(
    raw_data[:-delay],
    targets=temperature[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    shuffle=True,
    batch_size=batch_size,
    start_index=num_train_samples + num_val_samples)

**Inspecting the output of one of our datasets**

In [ ]:
for samples, targets in train_dataset:
    print("samples shape:", samples.shape)
    print("targets shape:", targets.shape)
    break

In [ ]:
for samples, targets in train_dataset:
    print("samples shape:", samples.shape)
    print("targets shape:", targets.shape)

    print(samples[0][0])
    print(targets[0])
    print(samples[0][1])
    print(targets[0])

    break

### Let's try a basic machine-learning model

In this model, we will use only the basic Dense layers. This will be our baseline model and we will show how RNNS can improve performance beating the baseline MAE score.

**Training and evaluating a densely connected model**

Mean Absolute Error is one of the common metrics used for regression tasks like temperature forecasting. Here's how you would compute MAE:

a. Use the trained model to make predictions on the testing dataset.

b. For each prediction, calculate the absolute difference between the predicted temperature and the actual temperature.

c. Take the average of all these absolute differences to get the Mean Absolute Error.

MAE represents the average magnitude of errors between predicted and actual temperatures. For example, if the MAE is 2 degrees Celsius, it means that, on average, the model's predictions deviate from the actual temperatures by 2 degrees Celsius. Lower MAE values indicate better performance, as they imply that the model's predictions are closer to the actual values.



In [ ]:
print(sequence_length)
print(raw_data.shape)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.Flatten()(inputs)
x = layers.Dense(16, activation="relu")(x)
outputs = layers.Dense(1)(x)
model = keras.Model(inputs, outputs)

callbacks = [
    keras.callbacks.ModelCheckpoint("jena_dense.keras",
                                    save_best_only=True)
]
model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
history = model.fit(train_dataset,
                    epochs=10,
                    validation_data=val_dataset,
                    callbacks=callbacks)

model = keras.models.load_model("jena_dense.keras")
print(f"Test MAE: {model.evaluate(test_dataset)[1]:.2f}")

**Plotting results**

In [ ]:
import matplotlib.pyplot as plt
loss = history.history["mae"]
val_loss = history.history["val_mae"]
epochs = range(1, len(loss) + 1)
plt.figure()
plt.plot(epochs, loss, "bo", label="Training MAE")
plt.plot(epochs, val_loss, "b", label="Validation MAE")
plt.title("Training and validation MAE")
plt.legend()
plt.show()

### Let's try a 1D convolutional model

Using a 1D convolutional neural network (CNN) for temperature forecasting involves applying convolutional layers to input sequences of temperature data.

Model Architecture:

1. Start with an input layer that takes sequences of temperature data as input.
2. Add one or more 1D convolutional layers to extract features from the input sequences. Each convolutional layer applies a set of filters across the input sequence, capturing patterns at different temporal scales.
3. Optionally, you can add pooling layers (e.g., MaxPooling1D) to downsample the output of the convolutional layers.
4. Flatten or use global pooling to convert the output of the convolutional layers into a 1D vector.
5. Add one or more fully connected layers (Dense layers) to perform the final regression prediction.
6. Optionally, include dropout layers for regularization to prevent overfitting.
7. The output layer should have a single neuron since temperature forecasting is a regression task, and it should use an appropriate activation function (e.g., linear activation for regression).

In [ ]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.Conv1D(8, 24, activation="relu")(inputs)
x = layers.MaxPooling1D(2)(x)
x = layers.Conv1D(8, 12, activation="relu")(x)
x = layers.MaxPooling1D(2)(x)
x = layers.Conv1D(8, 6, activation="relu")(x)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(1)(x)
model = keras.Model(inputs, outputs)

callbacks = [
    keras.callbacks.ModelCheckpoint("jena_conv.keras",
                                    save_best_only=True)
]
model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
history = model.fit(train_dataset,
                    epochs=10,
                    validation_data=val_dataset,
                    callbacks=callbacks)

model = keras.models.load_model("jena_conv.keras")
print(f"Test MAE: {model.evaluate(test_dataset)[1]:.2f}")

### A first recurrent baseline

**A simple LSTM-based model**

Using Long Short-Term Memory (LSTM) networks for temperature forecasting is a common approach due to their ability to capture long-term dependencies in sequential data.

ong Short-Term Memory (LSTM) networks are a type of recurrent neural network (RNN) designed to capture long-term dependencies in sequential data. The LSTM layer consists of memory cells and various gates that control the flow of information through the cell over time.

In [ ]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.LSTM(16)(inputs)
outputs = layers.Dense(1)(x)
model = keras.Model(inputs, outputs)

callbacks = [
    keras.callbacks.ModelCheckpoint("jena_lstm.keras",
                                    save_best_only=True)
]
model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
history = model.fit(train_dataset,
                    epochs=10,
                    validation_data=val_dataset,
                    callbacks=callbacks)

model = keras.models.load_model("jena_lstm.keras")
print(f"Test MAE: {model.evaluate(test_dataset)[1]:.2f}")

## Advanced use of recurrent neural networks

### Using recurrent dropout to fight overfitting

**Training and evaluating a dropout-regularized LSTM**

In [ ]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.LSTM(32, recurrent_dropout=0.25)(inputs)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)
model = keras.Model(inputs, outputs)

callbacks = [
    keras.callbacks.ModelCheckpoint("jena_lstm_dropout.keras",
                                    save_best_only=True)
]
model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
history = model.fit(train_dataset,
                    epochs=50,
                    validation_data=val_dataset,
                    callbacks=callbacks)

model = keras.models.load_model("jena_lstm_dropout.keras")
print(f"Test MAE: {model.evaluate(test_dataset)[1]:.2f}")

### Stacking recurrent layers

**Training and evaluating a dropout-regularized, stacked GRU model**

A Gated Recurrent Unit (GRU) layer is another type of recurrent neural network (RNN) layer that is similar to LSTM (Long Short-Term Memory) but with some variations in its architecture. GRU was introduced as a simpler alternative to LSTM, with fewer parameters and computational complexity, yet still capable of capturing long-term dependencies in sequential data.

In [ ]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.GRU(32, recurrent_dropout=0.5, return_sequences=True)(inputs)
x = layers.GRU(32, recurrent_dropout=0.5)(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)
model = keras.Model(inputs, outputs)

callbacks = [
    keras.callbacks.ModelCheckpoint("jena_stacked_gru_dropout.keras",
                                    save_best_only=True)
]
model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
history = model.fit(train_dataset,
                    epochs=50,
                    validation_data=val_dataset,
                    callbacks=callbacks)
model = keras.models.load_model("jena_stacked_gru_dropout.keras")
print(f"Test MAE: {model.evaluate(test_dataset)[1]:.2f}")

Compare the test MAE results of the 5 models above. Are you able to modify any of the networks to improve this performance?

## References

Chollet, F. (2017). Deep learning with python. Manning Publications.